# Notebook simplificada - Fondo + UNI

Flujo:
1) Extraccion de fondo simple (intensidad + blur)
2) Separacion nucleo vs citoplasma con embeddings UNI

In [ ]:
# CELDA 1: SETUP + MODELO

import sys
import subprocess
import numpy as np
import cv2
import matplotlib.pyplot as plt

print("⏳ Configurando entorno...\n")

packages = [
    "open-clip-torch==2.23.0",
    "transformers==4.35.2",
    "scikit-learn==1.4.2",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    print(f"✅ {pkg}")

from utils.model_factory import create_model
from utils.embeddings import get_all_patch_embeddings_from_image
from utils.coco_loader import list_coco_images, load_image_and_segmentations_from_coco

MODEL_NAME = "uni"
print(f"\n🔬 Cargando modelo: {MODEL_NAME}...")
model = create_model(MODEL_NAME)
print("✅ Setup completado")

In [ ]:
# CELDA 2: CARGAR IMAGEN

JSON_PATH = r"C:\Users\mngra\projects\AI\Pap\PAP_DATA\CCEDD\CCEDD-UTN-10imgs\train\_annotations.coco.json"
IMAGES_DIR = r"C:\Users\mngra\projects\AI\Pap\PAP_DATA\CCEDD\CCEDD-UTN-10imgs\train"

images_list = list_coco_images(JSON_PATH)
IMAGE_INDEX = 0
CATEGORY_IDS = [4, 5]

img_original, segmentations, fname, (H_orig, W_orig) = load_image_and_segmentations_from_coco(
    JSON_PATH, IMAGES_DIR,
    image_id=images_list[IMAGE_INDEX]["id"],
    block_size=224,
    category_ids=CATEGORY_IDS
)

TARGET_SIZE = (1376, 1020)
scale = min(TARGET_SIZE[0] / img_original.shape[1], TARGET_SIZE[1] / img_original.shape[0])
new_w = int(img_original.shape[1] * scale)
new_h = int(img_original.shape[0] * scale)
img_resized = cv2.resize(img_original, (new_w, new_h), interpolation=cv2.INTER_AREA)

if img_resized.ndim == 3:
    img = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
else:
    img = img_resized

H, W = img.shape[:2]
print(f"✅ {fname} | Dimensiones: {W}x{H}")

In [ ]:
# CELDA 3: PASO 1 (FONDO) + PASO 2 (UNI)

print("\n" + "="*60)
print("PASO 1: Extracción de fondo (simple)")
print("="*60)

blur = cv2.GaussianBlur(img, (51, 51), 0)
bg_thresh = np.percentile(blur, 85)
mask_fondo = blur >= bg_thresh
mask_tissue = ~mask_fondo

print(f"Umbral fondo (percentil 85): {bg_thresh:.2f}")
print(f"Fondo: {np.sum(mask_fondo)} px | Tejido: {np.sum(mask_tissue)} px")

print("\n" + "="*60)
print("PASO 2: UNI (núcleo vs citoplasma)")
print("="*60)

tile_size = 224
stride = 224

patches = get_all_patch_embeddings_from_image(
    img_resized, model, preprocess=None, tile_size=tile_size, stride=stride
)

# Parsear embeddings + coords de forma robusta
embeddings = None
coords = None

if isinstance(patches, dict):
    embeddings = patches.get("embeddings") or patches.get("embs")
    coords = patches.get("coords") or patches.get("positions")
elif isinstance(patches, list) and len(patches) > 0:
    if isinstance(patches[0], dict):
        embeddings = np.stack([p.get("embedding") or p.get("emb") for p in patches])
        coords = np.array([p.get("coords") or p.get("position") for p in patches])
    else:
        embeddings = np.array(patches)

if embeddings is None:
    raise RuntimeError("No se pudieron leer embeddings de UNI")

# Si no hay coords, reconstruir grilla
if coords is None:
    rows = (H - tile_size) // stride + 1
    cols = (W - tile_size) // stride + 1
    if len(embeddings) != rows * cols:
        raise RuntimeError("No se pudo inferir grilla de parches")
    coords = [(r * stride, c * stride) for r in range(rows) for c in range(cols)]

coords = np.array(coords)

from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=2, random_state=0, n_init=10)
labels = kmeans.fit_predict(embeddings)

# Determinar cluster de núcleos por menor intensidad media
means = []
for (y, x) in coords:
    patch = img[y:y+tile_size, x:x+tile_size]
    means.append(patch.mean())
means = np.array(means)

cluster_mean_intensity = {
    0: means[labels == 0].mean() if np.any(labels == 0) else 1e9,
    1: means[labels == 1].mean() if np.any(labels == 1) else 1e9,
}
nucleus_label = min(cluster_mean_intensity, key=cluster_mean_intensity.get)

mask_nucleos = np.zeros((H, W), dtype=np.uint8)
for (y, x), lab in zip(coords, labels):
    if lab == nucleus_label:
        mask_nucleos[y:y+tile_size, x:x+tile_size] = 1

# Restringir a tejido (no fondo)
mask_nucleos = (mask_nucleos > 0) & mask_tissue
mask_citoplasma = (~mask_nucleos) & mask_tissue

print(f"Núcleos (UNI): {np.sum(mask_nucleos)} px")
print(f"Citoplasma (UNI): {np.sum(mask_citoplasma)} px")

# VISUALIZACIÓN
img_color = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
img_color = cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0, 0].imshow(img, cmap="gray")
axes[0, 0].set_title("Imagen")
axes[0, 0].axis("off")

axes[0, 1].imshow(mask_fondo, cmap="Blues")
axes[0, 1].set_title("Fondo")
axes[0, 1].axis("off")

axes[1, 0].imshow(mask_nucleos, cmap="Reds")
axes[1, 0].set_title("Núcleos (UNI)")
axes[1, 0].axis("off")

axes[1, 1].imshow(mask_citoplasma, cmap="Greens")
axes[1, 1].set_title("Citoplasma (UNI)")
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()